In [0]:
from pyspark.sql.functions import col, count, min, max, avg
import matplotlib.pyplot as plt
import pandas as pd

# Läs in Gold-tabellerna
dim_event = spark.table("marathos_catalog.marathon_gold.dim_event")
dim_athlete = spark.table("marathos_catalog.marathon_gold.dim_athlete")
fct_results = spark.table("marathos_catalog.marathon_gold.fct_results")

# Läs in vyerna
vw_50km = spark.table("marathos_catalog.marathon_gold.vw_50km_races")
vw_100km = spark.table("marathos_catalog.marathon_gold.vw_100km_races")



In [0]:
print(" Gold layer översikt")
print(f"dim_event: {dim_event.count():,} rader")
print(f"dim_athlete: {dim_athlete.count():,} rader")
print(f"fct_results: {fct_results.count():,} rader")
print(f"vw_50km_races: {vw_50km.count():,} rader")
print(f"vw_100km_races: {vw_100km.count():,} rader")

In [0]:
print(spark.table("marathos_catalog.marathon_gold.fct_results").count())

In [0]:
print(" TOP 10 IDROTTARE MED FLEST LOPP")

# Räkna lopp per idrottare
lopp_per_idrottare = fct_results.groupBy("athlete_id").count()

# Hämta första förekomsten av land och kön
athlete_unique = dim_athlete.groupBy("athlete_id").agg(
    first("athlete_country").alias("athlete_country"),
    first("gender").alias("gender")
)

# Slå ihop och ta top 10
top10 = lopp_per_idrottare.join(athlete_unique, "athlete_id") \
    .select("athlete_id", "athlete_country", "gender", "count") \
    .orderBy("count", ascending=False) \
    .limit(10)

top10.display()

In [0]:

print("År med flest deltagare")

popular_years = df_silver.groupBy("event_year").count() \
    .orderBy("count", ascending=False) \
    .limit(10)

popular_years = popular_years.withColumnRenamed("event_year", "år")
popular_years = popular_years.withColumnRenamed("count", "deltagare")

popular_years.display()

In [0]:
# Läs in data
dim_athlete = spark.table("marathos_catalog.marathon_gold.dim_athlete")

# Topp 10 länder
top10 = dim_athlete.groupBy("athlete_country").count() \
    .orderBy("count", ascending=False).limit(10).toPandas()

fig, ax = plt.subplots(figsize=(10, 8))
ax.barh(top10["athlete_country"], top10["count"], color="teal", edgecolor="black")
ax.set_xlabel("Antal idrottare", fontsize=12)
ax.set_ylabel("Land", fontsize=12)
ax.set_title("Topp 10 länder med flest idrottare", fontsize=14)
ax.invert_yaxis()

# Lägg till siffror
for i, (country, count) in enumerate(zip(top10["athlete_country"], top10["count"])):
    ax.text(count + 500, i, f"{int(count):,}", va="center", fontsize=10)

plt.tight_layout()
plt.show()

In [0]:



# M blir Man, F blir Kvinna, X och null blir Icke binär
dim_athlete_clean = dim_athlete.withColumn(
    "gender_clean",
    when(col("gender") == "M", "Man")
    .when(col("gender") == "F", "Kvinna")
    .when(col("gender") == "X", "Icke binär")
    .when(col("gender").isNull(), "Icke binär")
    .otherwise("Annat")
)

# Räknar hur många det är av varje kön och gör om till pandas
gender = dim_athlete_clean.groupBy("gender_clean").count().toPandas()

# Skapar ett stapeldiagram
fig, ax = plt.subplots(figsize=(8, 6))
bars = ax.bar(gender["gender_clean"], gender["count"], color=["#3498db", "#e74c3c", "#95a5a6"], edgecolor="black")
ax.set_xlabel("Kön", fontsize=12)
ax.set_ylabel("Antal idrottare", fontsize=12)
ax.set_title("Könsfördelning bland idrottare", fontsize=14)

ax.get_yaxis().get_major_formatter().set_scientific(False)
ax.ticklabel_format(style='plain', axis='y')

# Lägger till siffror ovanför varje stapel
for bar, count in zip(bars, gender["count"]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5000, f"{int(count):,}", ha="center", fontweight="bold")


plt.show()